In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
# 1. File Paths
INPUT_FILE = Path(
    '/Users/cc-hoa/Desktop/毕业论文代码/results/step_03_UHI.xlsx'
)

INPUT_SHEET = "Full_Results"

INDICATOR_FILE = Path(
   '/Users/cc-hoa/Desktop/毕业论文代码/results/step_01_04_z_standardised_data.xlsx'
)

INDICATOR_SHEET = "Standardised_Data"

OUTPUT_DIR = Path(
    "/Users/cc-hoa/Desktop/毕业论文代码/results/"
    "Step_06_01_Weighting_Sensitivity"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read UHI Results and Standardised Indicators
# Check whether input files exist
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"UHI input file not found:\n{INPUT_FILE}"
    )

if not INDICATOR_FILE.exists():
    raise FileNotFoundError(
        f"Indicator input file not found:\n{INDICATOR_FILE}"
    )


# Read baseline UHI results
uhi_data = pd.read_excel(
    INPUT_FILE,
    sheet_name=INPUT_SHEET,
    engine="openpyxl",
)


# Read standardised indicator data
indicator_data = pd.read_excel(
    INDICATOR_FILE,
    sheet_name=INDICATOR_SHEET,
    engine="openpyxl",
)


# Indicators required for the weighting sensitivity analysis
indicator_columns = [
    "FUA_Code",
    "z_Population",
    "z_Total_GDP_PPP",
    "z_GDP_per_capita_PPP",
    "z_Road_Density",
    "z_Railway_Stations",
    "z_Airports",
    "z_Universities",
    "z_Hospitals",
]


# Check whether required columns exist
missing_indicator_cols = [
    col
    for col in indicator_columns
    if col not in indicator_data.columns
]

if missing_indicator_cols:
    raise ValueError(
        "Missing columns in standardised indicator data: "
        f"{missing_indicator_cols}"
    )


# Check merge key
if "FUA_Code" not in uhi_data.columns:
    raise ValueError(
        "FUA_Code is missing from the UHI results file."
    )


# Merge baseline UHI results with indicator Z-scores
df = uhi_data.merge(
    indicator_data[indicator_columns],
    on="FUA_Code",
    how="inner",
    validate="one_to_one",
)

# Final merge check
if df.empty:
    raise ValueError(
        "The merged dataframe is empty. "
        "Please check whether FUA_Code values match between the two files."
    )


print(f"UHI rows: {len(uhi_data)}")
print(f"Indicator rows: {len(indicator_data)}")
print(f"Merged rows: {len(df)}")
print("Data loaded successfully.")

UHI rows: 56
Indicator rows: 56
Merged rows: 56
Data loaded successfully.


In [9]:
# Define Alternative Dimension Weighting Scenarios
# Baseline model:
# four theoretical dimensions are equally weighted
df["UHI_equal_dimensions"] = (
    0.25 * df["DEM_star"]
    + 0.25 * df["ECO_star"]
    + 0.25 * df["TRA_star"]
    + 0.25 * df["FUN_star"]
)

# Alternative scenario 1:
# greater emphasis on urban size and economic capacity
df["UHI_scale_economy"] = (
    0.30 * df["DEM_star"]
    + 0.30 * df["ECO_star"]
    + 0.20 * df["TRA_star"]
    + 0.20 * df["FUN_star"]
)


# Alternative scenario 2:
# greater emphasis on transport connectivity
# and higher-order urban functions
df["UHI_connectivity_function"] = (
    0.20 * df["DEM_star"]
    + 0.20 * df["ECO_star"]
    + 0.30 * df["TRA_star"]
    + 0.30 * df["FUN_star"]
)


# Store scenario names and corresponding UHI columns
SCENARIOS = {
    "Equal dimensions":
        "UHI_equal_dimensions",

    "Scale and economy emphasis":
        "UHI_scale_economy",

    "Connectivity and function emphasis":
        "UHI_connectivity_function",
}


# Baseline specification used for comparison
BASELINE_COLUMN = "UHI_equal_dimensions"


# Check that all weighting schemes sum to 1
WEIGHT_SCHEMES = {
    "Equal dimensions": [0.25, 0.25, 0.25, 0.25],
    "Scale and economy emphasis": [0.30, 0.30, 0.20, 0.20],
    "Connectivity and function emphasis": [0.20, 0.20, 0.30, 0.30],
}

for scenario_name, weights in WEIGHT_SCHEMES.items():
    if not np.isclose(sum(weights), 1.0):
        raise ValueError(
            f"Weights for '{scenario_name}' do not sum to 1."
        )


print("Weighting scenarios created successfully.")

Weighting scenarios created successfully.


In [10]:
# Function for Ordered Tier Assignment
def assign_tiers(values, k=4):
    """
    Apply one-dimensional K-means clustering and reorder
    clusters from highest to lowest mean score as Tier 1 to Tier k.
    """

    values = np.asarray(values)

    if np.isnan(values).any():
        raise ValueError(
            "Missing values detected in UHI scores before Tier assignment."
        )

    model = KMeans(
        n_clusters=k,
        random_state=2026,
        n_init=100,
    )

    labels = model.fit_predict(
        values.reshape(-1, 1)
    )

    cluster_means = (
        pd.DataFrame({
            "Cluster": labels,
            "Score": values,
        })
        .groupby("Cluster")["Score"]
        .mean()
        .sort_values(ascending=False)
    )

    tier_mapping = {
        cluster: tier
        for tier, cluster in enumerate(
            cluster_means.index,
            start=1,
        )
    }

    return np.array([
        tier_mapping[label]
        for label in labels
    ])
# Calculate Rankings and Tiers for Each Scenario
FINAL_K = 4

for scenario_name, score_column in SCENARIOS.items():

    safe_name = (
        scenario_name
        .lower()
        .replace(" ", "_")
    )
# UHI ranking: highest score = Rank 1
    df[f"Rank_{safe_name}"] = (
        df[score_column]
        .rank(
            ascending=False,
            method="min",
        )
        .astype(int)
    )

# Re-run the baseline Tier method under each weighting scenario
    df[f"Tier_{safe_name}"] = assign_tiers(
        df[score_column],
        k=FINAL_K,
    )


# Baseline columns for later robustness comparisons
BASELINE_RANK = "Rank_equal_dimensions"
BASELINE_TIER = "Tier_equal_dimensions"


print("Rankings and Tier assignments created successfully.")

Rankings and Tier assignments created successfully.


In [14]:
#Compare Alternative Weighting Scenarios with Baseline
summary_rows = []

for scenario_name, score_column in SCENARIOS.items():

    # Skip the baseline itself
    if score_column == BASELINE_COLUMN:
        continue

    safe_name = (
        scenario_name
        .lower()
        .replace(" ", "_")
    )

    rank_column = f"Rank_{safe_name}"
    tier_column = f"Tier_{safe_name}"

# Absolute change in city rankings
    rank_difference = (
        df[rank_column]
        - df[BASELINE_RANK]
    ).abs()

# Number of cities assigned to a different Tier
    tier_changes = (
        df[tier_column]
        != df[BASELINE_TIER]
    ).sum()

    summary_rows.append({

        "Scenario": scenario_name,

        "Pearson":
            df[BASELINE_COLUMN].corr(
                df[score_column],
                method="pearson",
            ),

        "Spearman":
            df[BASELINE_COLUMN].corr(
                df[score_column],
                method="spearman",
            ),

        "Mean Rank Change":
            rank_difference.mean(),

        "Maximum Rank Change":
            rank_difference.max(),

        "Tier Changes":
            tier_changes,

        "Tier Change (%)":
            100 * tier_changes / len(df),

    })


#Create Summary Table
sensitivity_summary = (
    pd.DataFrame(summary_rows)
    .round({
        "Pearson": 3,
        "Spearman": 3,
        "Mean Rank Change": 2,
        "Maximum Rank Change": 0,
        "Tier Change (%)": 1,
    })
)

print("\n")
print("Table 6.1 Alternative Weighting Sensitivity Summary")
print(sensitivity_summary)

#Save to Excel
output_file = (
    OUTPUT_DIR /
    "Step_06_01_Alternative_Weighting_Summary.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
) as writer:

    sensitivity_summary.to_excel(
        writer,
        sheet_name="Weighting_Summary",
        index=False,
    )

print(f"\nResults saved to:\n{output_file}")



Table 6.1 Alternative Weighting Sensitivity Summary
                             Scenario  Pearson  Spearman  Mean Rank Change  \
0          Scale and economy emphasis    0.998     0.997              0.82   
1  Connectivity and function emphasis    0.998     0.995              1.04   

   Maximum Rank Change  Tier Changes  Tier Change (%)  
0                    4             2              3.6  
1                    4             2              3.6  

Results saved to:
/Users/cc-hoa/Desktop/毕业论文代码/results/Step_06_01_Weighting_Sensitivity/Step_06_01_Alternative_Weighting_Summary.xlsx


In [15]:
#Create City-Level Sensitivity Results
city_results = df[
    [
        "FUA_Code",
        "FUA_Name",
        "ISO3",
        "UHI_equal_dimensions",
        "Rank_equal_dimensions",
        "Tier_equal_dimensions",
    ]
].copy()


for scenario_name, score_column in SCENARIOS.items():

#Skip baseline
    if score_column == BASELINE_COLUMN:
        continue

    safe_name = (
        scenario_name
        .lower()
        .replace(" ", "_")
    )

    rank_column = f"Rank_{safe_name}"
    tier_column = f"Tier_{safe_name}"

# Alternative UHI score
    city_results[score_column] = df[score_column]

# Alternative city ranking
    city_results[rank_column] = df[rank_column]

# Signed ranking change:
# positive = lower position than baseline
# negative = higher position than baseline
    city_results[
        f"Rank_Change_{safe_name}"
    ] = (
        df[rank_column]
        - df[BASELINE_RANK]
    )

    # Absolute ranking change
    city_results[
        f"Absolute_Rank_Change_{safe_name}"
    ] = (
        city_results[
            f"Rank_Change_{safe_name}"
        ].abs()
    )

    # Alternative Tier
    city_results[tier_column] = df[tier_column]

    # Whether Tier membership changed
    city_results[
        f"Tier_Changed_{safe_name}"
    ] = (
        df[tier_column]
        != df[BASELINE_TIER]
    )

#Identify Cities with Tier Changes
tier_change_columns = [
    col
    for col in city_results.columns
    if col.startswith("Tier_Changed_")
]

tier_changed_cities = city_results[
    city_results[tier_change_columns].any(axis=1)
].copy()

#Save Final Weighting Sensitivity Results
OUTPUT_EXCEL = (
    OUTPUT_DIR
    / "Step_06_01_Alternative_Weighting_Sensitivity.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl",
) as writer:

    # Main table for dissertation reporting
    sensitivity_summary.to_excel(
        writer,
        sheet_name="Weighting_Summary",
        index=False,
    )

    # Full city-level diagnostic results
    city_results.to_excel(
        writer,
        sheet_name="City_Level_Results",
        index=False,
    )

    # Cities whose Tier classification changed
    tier_changed_cities.to_excel(
        writer,
        sheet_name="Tier_Changed_Cities",
        index=False,
    )

print("Alternative Weighting Sensitivity Analysis Completed")
print(
    f"Number of cities with at least one Tier change: "
    f"{len(tier_changed_cities)}"
)
print(f"\nResults saved to:\n{OUTPUT_EXCEL}")

Alternative Weighting Sensitivity Analysis Completed
Number of cities with at least one Tier change: 4

Results saved to:
/Users/cc-hoa/Desktop/毕业论文代码/results/Step_06_01_Weighting_Sensitivity/Step_06_01_Alternative_Weighting_Sensitivity.xlsx


In [17]:
#Sensitivity to Data Treatment
#canada
SOURCE_FILE = Path(
    '/Users/cc-hoa/Desktop/毕业论文代码/results/UHI_冻结建模数据_指标审计.xlsx'
)

BASELINE_RESULT_FILE = Path(
     '/Users/cc-hoa/Desktop/毕业论文代码/results/step_03_UHI.xlsx'
)

OUTPUT_DIR = Path(
   "/Users/cc-hoa/Desktop/毕业论文代码/results/"
    "Step_06_03_Data_Sensitivity"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [18]:
raw = pd.read_excel(
    SOURCE_FILE,
    sheet_name="Frozen_Model_Data",
    engine="openpyxl",
)

canada_audit = pd.read_excel(
    SOURCE_FILE,
    sheet_name="Canada_GDP_Audit",
    engine="openpyxl",
)

published_baseline = pd.read_excel(
    BASELINE_RESULT_FILE,
    sheet_name="Full_Results",
    engine="openpyxl",
)

In [20]:
# Re-run Complete UHI Pipeline
def rerun_uhi_pipeline(data):
    """
    Re-run the complete UHI construction workflow
    under an alternative data treatment.
    """

    model = data.copy()
#1. Log transformation
    model["t_population"] = np.log(
        model["Population"]
    )

    model["t_total_gdp"] = np.log(
        model["Total_GDP_PPP_USD_million"]
    )

    model["t_gdp_pc"] = np.log(
        model["GDP_per_capita_PPP_USD"]
    )

    model["t_road_density"] = np.log(
        model["Road_Density_km_per_km2"]
    )

    model["t_rail"] = np.log1p(
        model["Railway_Stations"]
    )

    model["t_airports"] = np.log1p(
        model["Airports"]
    )

    model["t_universities"] = np.log1p(
        model["Universities"]
    )

    model["t_hospitals"] = np.log1p(
        model["Hospitals"]
    )

#Indicator Z-standardisation
    transformed_columns = {
        "z_population": "t_population",
        "z_total_gdp": "t_total_gdp",
        "z_gdp_pc": "t_gdp_pc",
        "z_road_density": "t_road_density",
        "z_rail": "t_rail",
        "z_airports": "t_airports",
        "z_universities": "t_universities",
        "z_hospitals": "t_hospitals",
    }

    for z_column, transformed_column in transformed_columns.items():

        mean_value = model[transformed_column].mean()

        sd_value = model[transformed_column].std(
            ddof=1
        )

        if sd_value == 0:
            raise ValueError(
                f"Standard deviation is zero for "
                f"{transformed_column}."
            )

        model[z_column] = (
            model[transformed_column]
            - mean_value
        ) / sd_value

 #Construct dimension scores
    model["DEM_raw"] = model[
        "z_population"
    ]

    model["ECO_raw"] = model[
        [
            "z_total_gdp",
            "z_gdp_pc",
        ]
    ].mean(axis=1)

    model["TRA_raw"] = model[
        [
            "z_road_density",
            "z_rail",
            "z_airports",
        ]
    ].mean(axis=1)

    model["FUN_raw"] = model[
        [
            "z_universities",
            "z_hospitals",
        ]
    ].mean(axis=1)

#Re-standardise dimension scores
    for dimension in [
        "DEM",
        "ECO",
        "TRA",
        "FUN",
    ]:

        raw_column = f"{dimension}_raw"
        star_column = f"{dimension}_star"

        mean_value = model[raw_column].mean()

        sd_value = model[raw_column].std(
            ddof=1
        )

        if sd_value == 0:
            raise ValueError(
                f"Standard deviation is zero for "
                f"{raw_column}."
            )

        model[star_column] = (
            model[raw_column]
            - mean_value
        ) / sd_value
# Construct equally weighted UHI
    model["UHI"] = model[
        [
            "DEM_star",
            "ECO_star",
            "TRA_star",
            "FUN_star",
        ]
    ].mean(axis=1)
#Calculate UHI ranking
    model["UHI_rank"] = (
        model["UHI"]
        .rank(
            ascending=False,
            method="min",
        )
        .astype(int)
    )

    return model
# Re-run Tier Classification
def rerun_tier_classification(model, k=4):
    """
    Re-run one-dimensional K-means Tier classification
    using the same specification as the baseline model.
    """

    result = model.copy()

    if result["UHI"].isna().any():
        raise ValueError(
            "Missing UHI values detected before Tier classification."
        )

    clustering = KMeans(
        n_clusters=k,
        random_state=2026,
        n_init=100,
    )

    raw_labels = clustering.fit_predict(
        result[["UHI"]]
    )

# Order clusters by mean UHI:
# highest cluster = Tier 1
    cluster_means = (
        pd.DataFrame({
            "Cluster": raw_labels,
            "UHI": result["UHI"],
        })
        .groupby("Cluster")["UHI"]
        .mean()
        .sort_values(ascending=False)
    )

    label_to_tier = {
        cluster_label: tier
        for tier, cluster_label in enumerate(
            cluster_means.index,
            start=1,
        )
    }

    result["Tier"] = [
        label_to_tier[label]
        for label in raw_labels
    ]

    return result

In [22]:
#Reconstruct Baseline Model
baseline = rerun_uhi_pipeline(raw)

baseline = rerun_tier_classification(
    baseline,
    k=4,
)

baseline = baseline.rename(columns={
    "UHI": "UHI_Baseline",
    "UHI_rank": "Rank_Baseline",
    "Tier": "Tier_Baseline",
})

#Validate Reconstructed Baseline
baseline_check = baseline[
    [
        "FUA_Code",
        "UHI_Baseline",
    ]
].merge(
    published_baseline[
        [
            "FUA_Code",
            "UHI",
        ]
    ],
    on="FUA_Code",
    how="inner",
    validate="one_to_one",
)

baseline_check["Absolute_Difference"] = (
    baseline_check["UHI_Baseline"]
    - baseline_check["UHI"]
).abs()

max_baseline_difference = (
    baseline_check["Absolute_Difference"].max()
)

print("=" * 80)
print("Baseline Reconstruction Check")
print("=" * 80)
print(
    f"Maximum absolute UHI difference: "
    f"{max_baseline_difference:.12f}"
)
print()


# Alternative Data Treatment:
# Unscaled Canadian CMA GDP
unscaled_gdp = (
    canada_audit[
        [
            "FUA_Code",
            "CMA_GDP_PPP_USD_million_2021",
        ]
    ]
    .rename(columns={
        "CMA_GDP_PPP_USD_million_2021":
            "Unscaled_CMA_GDP"
    })
)

unscaled_data = raw.merge(
    unscaled_gdp,
    on="FUA_Code",
    how="left",
    validate="one_to_one",
)

canada_mask = (
    unscaled_data["ISO3"] == "CAN"
)

if unscaled_data.loc[
    canada_mask,
    "Unscaled_CMA_GDP"
].isna().any():
    raise ValueError(
        "Missing unscaled CMA GDP values for Canadian FUAs."
    )

unscaled_data.loc[
    canada_mask,
    "Total_GDP_PPP_USD_million"
] = unscaled_data.loc[
    canada_mask,
    "Unscaled_CMA_GDP"
]

unscaled = rerun_uhi_pipeline(
    unscaled_data
)

unscaled = rerun_tier_classification(
    unscaled,
    k=4,
)

unscaled = unscaled.rename(columns={
    "UHI": "UHI_Unscaled_CMA",
    "UHI_rank": "Rank_Unscaled_CMA",
    "Tier": "Tier_Unscaled_CMA",
})

#Compare Baseline with Unscaled CMA Scenario
baseline_vs_unscaled = baseline[
    [
        "FUA_Code",
        "FUA_Name",
        "ISO3",
        "UHI_Baseline",
        "Rank_Baseline",
        "Tier_Baseline",
    ]
].merge(
    unscaled[
        [
            "FUA_Code",
            "UHI_Unscaled_CMA",
            "Rank_Unscaled_CMA",
            "Tier_Unscaled_CMA",
        ]
    ],
    on="FUA_Code",
    how="inner",
    validate="one_to_one",
)

baseline_vs_unscaled["UHI_Difference"] = (
    baseline_vs_unscaled["UHI_Unscaled_CMA"]
    - baseline_vs_unscaled["UHI_Baseline"]
)

baseline_vs_unscaled["Rank_Change"] = (
    baseline_vs_unscaled["Rank_Unscaled_CMA"]
    - baseline_vs_unscaled["Rank_Baseline"]
)

baseline_vs_unscaled["Absolute_Rank_Change"] = (
    baseline_vs_unscaled["Rank_Change"].abs()
)

baseline_vs_unscaled["Tier_Changed"] = (
    baseline_vs_unscaled["Tier_Baseline"]
    != baseline_vs_unscaled["Tier_Unscaled_CMA"]
)


unscaled_pearson = (
    baseline_vs_unscaled[
        "UHI_Baseline"
    ].corr(
        baseline_vs_unscaled[
            "UHI_Unscaled_CMA"
        ],
        method="pearson",
    )
)

unscaled_spearman = (
    baseline_vs_unscaled[
        "UHI_Baseline"
    ].corr(
        baseline_vs_unscaled[
            "UHI_Unscaled_CMA"
        ],
        method="spearman",
    )
)

Baseline Reconstruction Check
Maximum absolute UHI difference: 0.000000000000



In [24]:
#Additional Sample Sensitivity:
#Exclude Canadian FUAs
non_canada_data = raw.loc[
    raw["ISO3"] != "CAN"
].copy()

# Re-run the full UHI pipeline without Canadian cities
exclude_canada = rerun_uhi_pipeline(
    non_canada_data
)

exclude_canada = rerun_tier_classification(
    exclude_canada,
    k=4,
)

exclude_canada = exclude_canada.rename(
    columns={
        "UHI": "UHI_Exclude_CAN",
        "UHI_rank": "Rank_Exclude_CAN",
        "Tier": "Tier_Exclude_CAN",
    }
)
# Prepare Comparable Baseline Sample
# Keep only non-Canadian cities in the baseline model
baseline_non_canada = baseline.loc[
    baseline["ISO3"] != "CAN",
    [
        "FUA_Code",
        "FUA_Name",
        "ISO3",
        "UHI_Baseline",
        "Tier_Baseline",
    ],
].copy()

# Re-rank baseline cities within the same non-Canadian sample
baseline_non_canada["Rank_Baseline_NonCAN"] = (
    baseline_non_canada["UHI_Baseline"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

#Compare Baseline with Canada-Exclusion Scenario
baseline_vs_exclusion = baseline_non_canada.merge(
    exclude_canada[
        [
            "FUA_Code",
            "UHI_Exclude_CAN",
            "Rank_Exclude_CAN",
            "Tier_Exclude_CAN",
        ]
    ],
    on="FUA_Code",
    how="inner",
    validate="one_to_one",
)

baseline_vs_exclusion["UHI_Difference"] = (
    baseline_vs_exclusion["UHI_Exclude_CAN"]
    - baseline_vs_exclusion["UHI_Baseline"]
)

baseline_vs_exclusion["Rank_Change"] = (
    baseline_vs_exclusion["Rank_Exclude_CAN"]
    - baseline_vs_exclusion["Rank_Baseline_NonCAN"]
)

baseline_vs_exclusion["Absolute_Rank_Change"] = (
    baseline_vs_exclusion["Rank_Change"].abs()
)

baseline_vs_exclusion["Tier_Changed"] = (
    baseline_vs_exclusion["Tier_Baseline"]
    != baseline_vs_exclusion["Tier_Exclude_CAN"]
)

# Correlation of continuous hierarchy scores
exclusion_pearson = (
    baseline_vs_exclusion[
        "UHI_Baseline"
    ].corr(
        baseline_vs_exclusion[
            "UHI_Exclude_CAN"
        ],
        method="pearson",
    )
)

exclusion_spearman = (
    baseline_vs_exclusion[
        "UHI_Baseline"
    ].corr(
        baseline_vs_exclusion[
            "UHI_Exclude_CAN"
        ],
        method="spearman",
    )
)

In [27]:
# Create Data Sensitivity Summary Table
sensitivity_summary = pd.DataFrame([
    {
        "Scenario":
            "Unscaled Canadian CMA GDP",

        "N":
            len(baseline_vs_unscaled),

        "Pearson r":
            unscaled_pearson,

        "Spearman ρ":
            unscaled_spearman,

        "Mean Rank Change":
            baseline_vs_unscaled[
                "Absolute_Rank_Change"
            ].mean(),

        "Maximum Rank Change":
            baseline_vs_unscaled[
                "Absolute_Rank_Change"
            ].max(),

        "Tier Changes (n)":
            baseline_vs_unscaled[
                "Tier_Changed"
            ].sum(),

        "Tier Changes (%)":
            100
            * baseline_vs_unscaled[
                "Tier_Changed"
            ].sum()
            / len(baseline_vs_unscaled),
    },

    {
        "Scenario":
            "Exclude Canada",

        "N":
            len(baseline_vs_exclusion),

        "Pearson r":
            exclusion_pearson,

        "Spearman ρ":
            exclusion_spearman,

        "Mean Rank Change":
            baseline_vs_exclusion[
                "Absolute_Rank_Change"
            ].mean(),

        "Maximum Rank Change":
            baseline_vs_exclusion[
                "Absolute_Rank_Change"
            ].max(),

        "Tier Changes (n)":
            baseline_vs_exclusion[
                "Tier_Changed"
            ].sum(),

        "Tier Changes (%)":
            100
            * baseline_vs_exclusion[
                "Tier_Changed"
            ].sum()
            / len(baseline_vs_exclusion),
    },
])


# Round values for reporting
sensitivity_summary = sensitivity_summary.round({
    "Pearson r": 3,
    "Spearman ρ": 3,
    "Mean Rank Change": 2,
    "Maximum Rank Change": 0,
    "Tier Changes (%)": 1,
})


print("\n")
print("Data Sensitivity Summary")
print(sensitivity_summary)

#Canadian City-Level Changes
canadian_city_changes = (
    baseline_vs_unscaled.loc[
        baseline_vs_unscaled["ISO3"] == "CAN",
        [
            "FUA_Code",
            "FUA_Name",
            "UHI_Baseline",
            "UHI_Unscaled_CMA",
            "Rank_Baseline",
            "Rank_Unscaled_CMA",
            "Rank_Change",
            "Tier_Baseline",
            "Tier_Unscaled_CMA",
            "Tier_Changed",
        ]
    ]
    .sort_values(
        "Rank_Baseline"
    )
    .copy()
)


print(
    f"\nCanadian FUAs affected: "
    f"{len(canadian_city_changes)}"
)

print(
    "Canadian FUAs with Tier changes: "
    f"{canadian_city_changes['Tier_Changed'].sum()}"
)



Data Sensitivity Summary
                    Scenario   N  Pearson r  Spearman ρ  Mean Rank Change  \
0  Unscaled Canadian CMA GDP  56      1.000       1.000              0.11   
1             Exclude Canada  48      0.999       0.998              0.54   

   Maximum Rank Change  Tier Changes (n)  Tier Changes (%)  
0                    1                 0               0.0  
1                    3                23              47.9  

Canadian FUAs affected: 8
Canadian FUAs with Tier changes: 0


In [29]:
#Identify Cities with Tier Changes
unscaled_tier_changes = (
    baseline_vs_unscaled.loc[
        baseline_vs_unscaled["Tier_Changed"]
    ]
    .copy()
)

exclusion_tier_changes = (
    baseline_vs_exclusion.loc[
        baseline_vs_exclusion["Tier_Changed"]
    ]
    .copy()
)
#Save Data Sensitivity Results
OUTPUT_EXCEL = (
    OUTPUT_DIR
    / "Step_06_03_Data_Sensitivity.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl",
) as writer:

# Main dissertation summary table
    sensitivity_summary.to_excel(
        writer,
        sheet_name="Data_Sensitivity_Summary",
        index=False,
    )

# Definition-change scenario:
# baseline vs unscaled Canadian CMA GDP
    baseline_vs_unscaled.to_excel(
        writer,
        sheet_name="Baseline_vs_Unscaled",
        index=False,
    )

# Canadian FUAs under alternative GDP treatment
    canadian_city_changes.to_excel(
        writer,
        sheet_name="Canadian_Cities",
        index=False,
    )

# Any Tier changes under alternative GDP treatment
    unscaled_tier_changes.to_excel(
        writer,
        sheet_name="Unscaled_Tier_Changes",
        index=False,
    )

# Sample-change scenario:
# baseline vs Canada exclusion
    baseline_vs_exclusion.to_excel(
        writer,
        sheet_name="Baseline_vs_Exclusion",
        index=False,
    )

# Cities whose Tier changes after excluding Canada
    exclusion_tier_changes.to_excel(
        writer,
        sheet_name="Exclusion_Tier_Changes",
        index=False,
    )

# Internal validation:
# reconstructed baseline vs published Step 03 result
    baseline_check.to_excel(
        writer,
        sheet_name="Baseline_Check",
        index=False,
    )

print("Data Sensitivity Analysis Completed")

print(
    f"Tier changes under unscaled CMA GDP: "
    f"{len(unscaled_tier_changes)}"
)

print(
    f"Tier changes after excluding Canada: "
    f"{len(exclusion_tier_changes)}"
)

print(f"\nResults saved to:\n{OUTPUT_EXCEL}")

Data Sensitivity Analysis Completed
Tier changes under unscaled CMA GDP: 0
Tier changes after excluding Canada: 23

Results saved to:
/Users/cc-hoa/Desktop/毕业论文代码/results/Step_06_03_Data_Sensitivity/Step_06_03_Data_Sensitivity.xlsx
